In [3]:
# Cell 1 — Extract all FAQs
import json

import pandas as pd

selected_schemes_file = "G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\golden_dataset\\selected_schemes.json"

with open(selected_schemes_file, "r", encoding="utf-8") as file:
    selected_schemes = json.load(file)

faq_records = []

for scheme in selected_schemes:

    scheme_id = scheme.get("scheme_id")
    scheme_name = scheme.get("name")

    faqs = scheme.get("faqs", [])

    for index, faq in enumerate(faqs):

        faq_records.append({
            "scheme_id": scheme_id,
            "scheme_name": scheme_name,
            "faq_id": f"{scheme_id}_FAQ_{index + 1}",
            "question": faq.get("question", ""),
            "answer": faq.get("answer", ""),
            "language": "unknown",
            "verification_status": "pending"
        })

faq_df = pd.DataFrame(faq_records)

print("Total FAQ records:", len(faq_df))

faq_df.head()

Total FAQ records: 6485


,scheme_id,scheme_name,faq_id,question,answer,language,verification_status
0,-Re2F5gB5uF80_MGHY9d,Chief Minister Floriculture Mission,-Re2F5gB5uF80_MGHY9d_FAQ_1,What About Floriculture Mission?,A farm based income generating enterprise havi...,unknown,pending
1,-Re2F5gB5uF80_MGHY9d,Chief Minister Floriculture Mission,-Re2F5gB5uF80_MGHY9d_FAQ_2,When was horticulture mission launched?,National Horticulture Mission (NHM) is an Indi...,unknown,pending
2,-Re2F5gB5uF80_MGHY9d,Chief Minister Floriculture Mission,-Re2F5gB5uF80_MGHY9d_FAQ_3,Who is the Father of floriculture?,Marigowda(Maragowdanahalli Hombegowda marigowd...,unknown,pending
3,-Re2F5gB5uF80_MGHY9d,Chief Minister Floriculture Mission,-Re2F5gB5uF80_MGHY9d_FAQ_4,Who is the father of floriculture in India?,"Being the doyen of horticulture, Dr. Chadha is...",unknown,pending
4,-Re2F5gB5uF80_MGHY9d,Chief Minister Floriculture Mission,-Re2F5gB5uF80_MGHY9d_FAQ_5,Which state is first in floriculture?,"Karnataka is the leader in floriculture, accou...",unknown,pending


In [4]:
# Check missing information
print("Total FAQs:", len(faq_df))

print(
    "Missing questions:",
    faq_df["question"].isna().sum()
    + (faq_df["question"].astype(str).str.strip() == "").sum()
)

print(
    "Missing answers:",
    faq_df["answer"].isna().sum()
    + (faq_df["answer"].astype(str).str.strip() == "").sum()
)

print(
    "Unique schemes covered:",
    faq_df["scheme_id"].nunique()
)

Total FAQs: 6485
Missing questions: 0
Missing answers: 0
Unique schemes covered: 591


In [5]:
# Analyze FAQs per scheme
faq_distribution = (
    faq_df.groupby(["scheme_id", "scheme_name"])
    .size()
    .reset_index(name="faq_count")
)

print(faq_distribution["faq_count"].describe())

faq_distribution.sort_values(
    "faq_count",
    ascending=False
).head(10)

count    591.000000
mean      10.972927
std        3.428091
min        1.000000
25%        9.000000
50%       10.000000
75%       12.000000
max       34.000000
Name: faq_count, dtype: float64


,scheme_id,scheme_name,faq_count
455,lxe0F5gB5uF80_MGEY9U,"AICTE - Scheme For Promoting Interests, Creati...",34
334,ZxezF5gB5uF80_MGi49l,State Post-metric Scholarship For Disabled,25
18,2cazF5gB0AGc0F0ikDMK,Homes For Intellectually Impaired Persons,25
508,r6HwF5gBXu-iUKXqtXgI,Chief Ministergi Artistesinggi Tengbang (CMAT)...,24
589,zxf9F5gB5uF80_MGwZNn,Goa (Technical/ Vocational Training To Sex Wor...,24
13,1sazF5gB0AGc0F0ijzO3,Financial Assistance To Disabled For Self Empl...,23
269,SqG7F5gBXu-iUKXq03ZE,Marriage Assistance (O.B.O.C.W.W.B),23
91,BMdkwJsB0AGc0F0iJACM,The Meghalaya Tribal Students’ Book Grant (Pos...,23
328,ZMa2F5gB0AGc0F0iHzSX,National Awards For Empowerment Of Persons Wit...,22
36,4qKUA58BXu-iUKXqH1nj,Investment Promotion Scheme (IPS) 2022- for Th...,22


In [ ]:
# Save FAQ dataset
from pathlib import Path

FAQ_OUTPUT = Path("data/golden_dataset")
FAQ_OUTPUT.mkdir(parents=True, exist_ok=True)

faq_df.to_csv(
    FAQ_OUTPUT / "original_faq_candidates.csv",
    index=False,
    encoding="utf-8-sig"
)

faq_df.to_json(
    FAQ_OUTPUT / "original_faq_candidates.json",
    orient="records",
    indent=4,
    force_ascii=False
)

print("FAQ candidates saved successfully!")

In [6]:
schemes_with_faqs = set(faq_df["scheme_id"].unique())

schemes_without_faqs = [
    scheme
    for scheme in selected_schemes
    if scheme["scheme_id"] not in schemes_with_faqs
]

print("Schemes without FAQs:", len(schemes_without_faqs))

for scheme in schemes_without_faqs:
    print(
        scheme.get("scheme_id"),
        scheme.get("name")
    )

Schemes without FAQs: 1
Lhh835sB5uF80_MGQlsH Mukhya Mantrir Eti koli duti paat 


In [7]:
# We can generate FAQs for Mukhya Mantrir Eti Koli Duti Paat using an LLM, based on its complete scheme JSON.
# Inspect available scheme information
missing_scheme = next(
    scheme for scheme in selected_schemes
    if scheme["scheme_id"] == "Lhh835sB5uF80_MGQlsH"
)

print("Scheme:", missing_scheme["name"])
print("ID:", missing_scheme["scheme_id"])

fields_to_check = [
    "brief_description",
    "details",
    "benefits",
    "eligibility",
    "exclusions",
    "documents_required",
    "application_process"
]

for field in fields_to_check:
    value = missing_scheme.get(field)
    print(f"\n{field}:")
    print(value if value else "Not available")

Scheme: Mukhya Mantrir Eti koli duti paat 
ID: Lhh835sB5uF80_MGQlsH

brief_description:
Mukhya Mantrir Eti koli duti paat also referred to as the Assam Tea Garden Workers' Financial Assistance Scheme 2025. This initiative marks the 200th-year celebration of Assam Tea and aims to provide one-time financial assistance to tea garden workers in recognition of their invaluable contribution.

details:
Mukhya Mantrir Eti koli duti paat (মুখ্যমন্ত্ৰীৰ এটি কলি, দুটি পাত), also referred to as the Assam Tea Garden Workers' Financial Assistance Scheme 2025, is launched by the Government of Assam to recognise the contribution of tea garden workers in the growth of the Tea Industry in the state. This initiative marks the 200th year celebration of Assam Tea and aims to provide one-time financial assistance to tea garden workers in recognition of their invaluable contribution, thereby supporting their socio-economic well-being.
The primary objectives of this scheme is:
a. To provide one-time financial

In [8]:
# Prepare scheme context
# We will pass only relevant information to the LLM.
import json

scheme_context = {
    field: missing_scheme.get(field)
    for field in fields_to_check
    if missing_scheme.get(field)
}

scheme_context["name"] = missing_scheme.get("name")
scheme_context["scheme_id"] = missing_scheme.get("scheme_id")
scheme_context["categories"] = missing_scheme.get("categories")
scheme_context["ministry"] = missing_scheme.get("ministry")

In [9]:
# Generate FAQs using Groq
# We'll use the Groq SDK directly.

import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

MODEL = os.getenv(
    "GROQ_MODEL",
    "openai/gpt-oss-120b"
)

In [10]:
# Create the FAQ generation prompt
# We will generate approximately 8–10 useful FAQs, covering different aspects of the scheme.

prompt = f"""
You are an expert government scheme information assistant
working on the SEVA AI project.

Generate 8 to 10 realistic FAQs for the following government scheme.

SCHEME INFORMATION:

{json.dumps(scheme_context, ensure_ascii=False, indent=2)}

INSTRUCTIONS:

1. Generate questions that ordinary citizens may ask.
2. Cover eligibility, benefits, application process,
   required documents and exclusions wherever information exists.
3. Do not invent any scheme rules, financial amounts,
   dates, eligibility conditions or documents.
4. Every answer must be supported by the supplied information.
5. Do not generate questions for information that is unavailable.
6. Avoid duplicate or semantically similar questions.
7. Use the same language as the source scheme information.
8. Keep answers informative but concise.
9. Include supporting evidence from the supplied scheme data.
10. If the available information is insufficient for 8 FAQs,
    generate fewer rather than inventing information.

Return valid JSON only:

{{
  "faqs": [
    {{
      "question": "Question",
      "answer": "Source-grounded answer",
      "question_type": "eligibility",
      "evidence": [
        {{
          "source_field": "eligibility",
          "quote": "Supporting text from source"
        }}
      ]
    }}
  ]
}}

Allowed question types:
eligibility, benefits, documents, application,
exclusions, general_information, other.
"""

In [11]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "You generate accurate, source-grounded "
                "government scheme FAQs. Never invent facts."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.1,
    max_tokens=5000
)

llm_output = response.choices[0].message.content

print(llm_output)

{
  "faqs": [
    {
      "question": "Who can apply for the Mukhya Mantrir Eti koli duti paat scheme?",
      "answer": "Any permanent resident of Assam who is currently residing in the state, is a registered tea garden worker or employee (excluding tea garden management), and holds an active Aadhaar‑linked bank account in a scheduled commercial bank.",
      "question_type": "eligibility",
      "evidence": [
        {
          "source_field": "eligibility",
          "quote": "• a permanent resident of Assam and presently residing in Assam\n• registered tea garden worker or employee, excluding TE management, engaged in an officially recognised Tea Garden of Assam\n• hold an active AADHAAR-linked bank account in a scheduled commercial bank."
        }
      ]
    },
    {
      "question": "What financial assistance will beneficiaries receive?",
      "answer": "Each eligible tea garden worker will receive a one‑time cash assistance of ₹5,000.",
      "question_type": "benefits",
  

In [12]:
# Parse generated FAQs
import re

clean_output = re.sub(
    r"^```(?:json)?\s*|\s*```$",
    "",
    llm_output.strip()
)

generated_data = json.loads(clean_output)

generated_faqs = generated_data["faqs"]

print("Generated FAQs:", len(generated_faqs))

generated_faqs[:2]

Generated FAQs: 9


[{'question': 'Who can apply for the Mukhya Mantrir Eti koli duti paat scheme?',
  'answer': 'Any permanent resident of Assam who is currently residing in the state, is a registered tea garden worker or employee (excluding tea garden management), and holds an active Aadhaar‑linked bank account in a scheduled commercial bank.',
  'question_type': 'eligibility',
  'evidence': [{'source_field': 'eligibility',
    'quote': '• a permanent resident of Assam and presently residing in Assam\n• registered tea garden worker or employee, excluding TE management, engaged in an officially recognised Tea Garden of Assam\n• hold an active AADHAAR-linked bank account in a scheduled commercial bank.'}]},
 {'question': 'What financial assistance will beneficiaries receive?',
  'answer': 'Each eligible tea garden worker will receive a one‑time cash assistance of ₹5,000.',
  'question_type': 'benefits',
  'evidence': [{'source_field': 'benefits',
    'quote': 'One-time financial assistance of ₹5000 to tea

In [13]:
# Convert to SEVA AI FAQ records
new_faq_records = []

for index, faq in enumerate(generated_faqs, start=1):

    new_faq_records.append({

        "scheme_id": missing_scheme["scheme_id"],
        "scheme_name": missing_scheme["name"],

        "faq_id": (
            f"{missing_scheme['scheme_id']}_LLM_FAQ_{index}"
        ),

        "question": faq["question"],
        "answer": faq["answer"],

        "question_type": faq.get(
            "question_type",
            "other"
        ),

        "evidence": faq.get("evidence", []),

        "question_source": "llm_generated",

        "verification_status": "pending"
    })

generated_faq_df = pd.DataFrame(new_faq_records)

generated_faq_df

,scheme_id,scheme_name,faq_id,question,answer,question_type,evidence,question_source,verification_status
0,Lhh835sB5uF80_MGQlsH,Mukhya Mantrir Eti koli duti paat,Lhh835sB5uF80_MGQlsH_LLM_FAQ_1,Who can apply for the Mukhya Mantrir Eti koli ...,Any permanent resident of Assam who is current...,eligibility,"[{'source_field': 'eligibility', 'quote': '• a...",llm_generated,pending
1,Lhh835sB5uF80_MGQlsH,Mukhya Mantrir Eti koli duti paat,Lhh835sB5uF80_MGQlsH_LLM_FAQ_2,What financial assistance will beneficiaries r...,Each eligible tea garden worker will receive a...,benefits,"[{'source_field': 'benefits', 'quote': 'One-ti...",llm_generated,pending
2,Lhh835sB5uF80_MGQlsH,Mukhya Mantrir Eti koli duti paat,Lhh835sB5uF80_MGQlsH_LLM_FAQ_3,What is the main purpose of the scheme?,The scheme aims to recognise the contribution ...,general_information,"[{'source_field': 'details', 'quote': 'This in...",llm_generated,pending
3,Lhh835sB5uF80_MGQlsH,Mukhya Mantrir Eti koli duti paat,Lhh835sB5uF80_MGQlsH_LLM_FAQ_4,Are tea garden owners or management eligible f...,"No. Owners, management of tea gardens, and own...",exclusions,"[{'source_field': 'exclusions', 'quote': 'Owne...",llm_generated,pending
4,Lhh835sB5uF80_MGQlsH,Mukhya Mantrir Eti koli duti paat,Lhh835sB5uF80_MGQlsH_LLM_FAQ_5,How will the financial assistance be disbursed...,"After verification and digitisation, a cheque ...",application,"[{'source_field': 'application_process', 'quot...",llm_generated,pending
5,Lhh835sB5uF80_MGQlsH,Mukhya Mantrir Eti koli duti paat,Lhh835sB5uF80_MGQlsH_LLM_FAQ_6,What are the steps to apply for the scheme?,1. Tea garden management submits a list of wor...,application,"[{'source_field': 'application_process', 'quot...",llm_generated,pending
6,Lhh835sB5uF80_MGQlsH,Mukhya Mantrir Eti koli duti paat,Lhh835sB5uF80_MGQlsH_LLM_FAQ_7,What documents or information are required to ...,The application requires (a) a list of tea gar...,documents,"[{'source_field': 'application_process', 'quot...",llm_generated,pending
7,Lhh835sB5uF80_MGQlsH,Mukhya Mantrir Eti koli duti paat,Lhh835sB5uF80_MGQlsH_LLM_FAQ_8,Can a worker who has already received cash ass...,No. Workers who have already received similar ...,exclusions,"[{'source_field': 'exclusions', 'quote': 'Indi...",llm_generated,pending
8,Lhh835sB5uF80_MGQlsH,Mukhya Mantrir Eti koli duti paat,Lhh835sB5uF80_MGQlsH_LLM_FAQ_9,Is a government employee working in a tea gard...,No. Government employees are listed among the ...,exclusions,"[{'source_field': 'exclusions', 'quote': 'Gove...",llm_generated,pending


In [14]:
# Save generated FAQs
output_file = (
    "G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\golden_dataset\\generated_faqs_missing_scheme.csv"
)

generated_faq_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", output_file)

Saved: G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\generated_faqs_missing_scheme.csv
